<a href="https://colab.research.google.com/github/ritan612/S-AES-OFB/blob/main/IN410.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Initialization

In [ ]:
import numpy as np
import random

def initializing_IV():
  IV = random.getrandbits(16)

  # Le '016b' signifie : format binaire (b), sur 16 caractères (16), rempli de zéros (0)
  print(f"{IV:016b}")

initializing_IV()
while True:
    key = input("Enter 16-bit key: ")

    # Check length and allowed characters
    if len(key) == 16 and all(bit in '01' for bit in key):
        break
    else:
        print("Invalid input! Please enter exactly 16 bits (only 0 and 1).")

print("Valid key:", key)

0101101000000100
Enter 16-bit key: 1111111100000000
Valid key: 1111111100000000


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
uploaded = files.upload()
uploaded_path = list(uploaded.keys())[0]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving Heelo World.txt to Heelo World (1).txt


In [ ]:
def file_to_bits_array(file_path, chunk_size=8192):
  bits_array = []

  with open(file_path, "rb") as f:
      while chunk := f.read(chunk_size):
          for byte in chunk:
              bits = f"{byte:08b}"
              for bit in bits:
                  bits_array.append(int(bit))
  return bits_array

bits = file_to_bits_array(uploaded_path)

print(bits[:64])  # first 64 bits
print(len(bits))   # total number of bits


[0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1]
88


In [ ]:
from itertools import batched
def batched_with_padding(iterable, batch_size, fill_value=0):
     for batch in batched(iterable, batch_size):
         yield batch + (fill_value,) * (batch_size - len(batch))

for batch in batched_with_padding(bits, 16):
     print(batch)

(0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1)
(0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0)
(0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0)
(0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1)
(0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0)
(0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)


In [ ]:
SBOX = [
    0x9, 0x4, 0xA, 0xB,
    0xD, 0x1, 0x8, 0x5,
    0x6, 0x2, 0x0, 0x3,
    0xC, 0xE, 0xF, 0x7
]

RCON1 = 0x80
RCON2 = 0x30

def sub_nib(byte):
    return (SBOX[byte >> 4] << 4) | SBOX[byte & 0x0F]

def rot_nib(byte):
    return ((byte << 4) | (byte >> 4)) & 0xFF

def key_expansion(key):
    w0 = (key >> 8) & 0xFF
    w1 = key & 0xFF

    w2 = w0 ^ sub_nib(rot_nib(w1)) ^ RCON1
    w3 = w2 ^ w1
    w4 = w2 ^ sub_nib(rot_nib(w3)) ^ RCON2
    w5 = w4 ^ w3

    K0 = (w0 << 8) | w1
    K1 = (w2 << 8) | w3
    K2 = (w4 << 8) | w5

    return K0, K1, K2

def add_round_key(state, key):
    return state ^ key

def sub_nibbles(state):
    return (
        (SBOX[(state >> 12) & 0xF] << 12) |
        (SBOX[(state >> 8) & 0xF] << 8) |
        (SBOX[(state >> 4) & 0xF] << 4) |
        SBOX[state & 0xF]
    )

def shift_rows(state):
    return (
        (state & 0xF000) |
        (state & 0x0F00) |
        ((state & 0x000F) << 4) |
        ((state & 0x00F0) >> 4)
    )

def gf_mult(a, b):
    p = 0
    for _ in range(4):
        if b & 1:
            p ^= a
        carry = a & 0x8
        a <<= 1
        if carry:
            a ^= 0b10011
        b >>= 1
    return p & 0xF

def mix_columns(state):
    s0 = (state >> 12) & 0xF
    s1 = (state >> 8) & 0xF
    s2 = (state >> 4) & 0xF
    s3 = state & 0xF

    t0 = gf_mult(s0, 1) ^ gf_mult(s2, 4)
    t2 = gf_mult(s0, 4) ^ gf_mult(s2, 1)
    t1 = gf_mult(s1, 1) ^ gf_mult(s3, 4)
    t3 = gf_mult(s1, 4) ^ gf_mult(s3, 1)

    return (t0 << 12) | (t1 << 8) | (t2 << 4) | t3

def encrypt(plaintext, key):
    K0, K1, K2 = key_expansion(key)

    state = add_round_key(plaintext, K0)

    # Round 1
    state = sub_nibbles(state)
    state = shift_rows(state)
    state = mix_columns(state)
    state = add_round_key(state, K1)

    # Round 2
    state = sub_nibbles(state)
    state = shift_rows(state)
    state = add_round_key(state, K2)

    return state

key = int("1010011100111011", 2)
plaintext = int("1101011100101000", 2)

cipher = encrypt(plaintext, key)

print("Ciphertext:", format(cipher, '016b'))

Ciphertext: 1000111000011110
